# **Training**

In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import timm
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import f1_score as sk_f1

torch.manual_seed(42)
np.random.seed(42)

# =========================
# CONFIGURATION
# =========================
SIZE_MODE    = "384"
POOLING_MODE = "gated_attention"

# ── Read-only inputs (Google Drive, do not write under these paths) ──────────
TRAIN_CSV     = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/train_dataset.csv"
VAL_CSV       = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/val_dataset.csv"
TEST_CSV      = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/test_dataset.csv"
BACKBONE_CKPT = "/content/drive/MyDrive/FYP/natural foci ground truths/backbone models/CELL_ONLY_foci_dino_backbone_DINOv03_v01.pth"

# ── Writable root for this revision (everything we write goes under here) ───
PAPER_REV_ROOT = "/content/drive/MyDrive/FYP/paper_revision_v01"
MODELS_DIR     = os.path.join(PAPER_REV_ROOT, "models")
PLOTS_DIR      = os.path.join(PAPER_REV_ROOT, "training_plots")

CFG = {
    "batch_size": 256,
    "epochs":     15,

    "lr_head": 1e-3,
    "wd":      1e-4,

    "train_csv": TRAIN_CSV,
    "val_csv":   VAL_CSV,

    "backbone_ckpt": BACKBONE_CKPT,
    "save_path":     os.path.join(MODELS_DIR, "baseline_model.pth"),
    "output_dir":    PLOTS_DIR,
    "device":        torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

if SIZE_MODE == "224":
    CFG.update({"image_size": 224, "backbone_name": "vit_tiny_patch16_224"})
elif SIZE_MODE == "384":
    CFG.update({"image_size": 384, "backbone_name": "vit_tiny_patch16_384"})

DEVICE = CFG["device"]
print(f"--- ABLATION D: No pos_weight (Plain BCE Loss) ---")
print(f"Pooling Strategy : {POOLING_MODE.upper()}")
print(f"Backbone Init    : DINO pretrained")
print(f"Backbone         : Fully frozen for all {CFG['epochs']} epochs")
print(f"Loss             : BCEWithLogitsLoss — NO pos_weight")
print(f"Writable root    : {PAPER_REV_ROOT}")

# =========================
# DATA AUGMENTATIONS & DATASET
# =========================

NORM_MEAN = [0.2251, 0.2251, 0.2251]
NORM_STD  = [0.2375, 0.2375, 0.2375]

def get_transforms(img_size, is_train=True):
    if is_train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.RandomApply([transforms.ColorJitter(brightness=0.1, contrast=0.1)], p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
        ])
    else:
        return transforms.Compose([
            transforms.Resize((img_size, img_size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
        ])

class FociDataset(Dataset):
    def __init__(self, csv_path, img_size=224, is_train=True):
        self.df        = pd.read_csv(csv_path)
        self.transform = get_transforms(img_size, is_train)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img_p = str(row['image_path'])
            img   = Image.open(img_p).convert("RGB")
            x     = self.transform(img)
            y     = float(row["label"])
            return x, torch.tensor(y).float()
        except Exception: return None

def safe_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    xs, ys = zip(*batch)
    return torch.stack(xs), torch.stack(ys)

In [ ]:
# =========================
# MODEL
# =========================
class FociViT_Baseline(nn.Module):
    def __init__(self, backbone_name, backbone_ckpt, img_size=224, pooling_mode="gated_attention"):
        super().__init__()
        self.pooling_mode = pooling_mode

        self.backbone = timm.create_model(
            backbone_name, pretrained=False,
            num_classes=0, global_pool="", img_size=img_size
        )

        if os.path.exists(backbone_ckpt):
            sd = torch.load(backbone_ckpt, map_location="cpu")
            self.backbone.load_state_dict(
                {k.replace("backbone.", "").replace("vit.", ""): v for k, v in sd.items()},
                strict=False
            )
            print("Loaded DINO Backbone Weights.")

        for param in self.backbone.parameters():
            param.requires_grad = False
        print("Backbone fully frozen for all epochs.")

        with torch.no_grad():
            dim = self.backbone.forward_features(
                torch.zeros(1, 3, img_size, img_size)
            ).shape[-1]

        if self.pooling_mode == "gated_attention":
            hidden_dim       = 128
            self.head_norm   = nn.LayerNorm(dim)
            self.attention_V = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Tanh())
            self.attention_U = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Sigmoid())
            self.attention_w = nn.Linear(hidden_dim, 1)

        self.classifier = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1)
        )

    def forward(self, x):
        feats = self.backbone.forward_features(x)
        if isinstance(feats, dict): feats = feats["x"]
        patch_tokens = feats[:, 1:, :]

        if self.pooling_mode == "gated_attention":
            normed_patches  = self.head_norm(patch_tokens)
            A_V             = self.attention_V(normed_patches)
            A_U             = self.attention_U(normed_patches)
            A_raw           = self.attention_w(A_V * A_U).squeeze(-1)
            A               = torch.softmax(A_raw, dim=1)
            pooled_features = torch.sum(patch_tokens * A.unsqueeze(-1), dim=1)
        else:
            pooled_features = feats[:, 0, :]

        logits = self.classifier(pooled_features).squeeze(-1)
        return logits


In [ ]:
# =========================
# MULTI-SEED TRAINING LOOP
# =========================

import random

N_SEEDS = 5

def set_all_seeds(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)


def train_one_seed(seed):
    print("\n" + "#"*60)
    print(f"  Training seed {seed}")
    print("#"*60)

    set_all_seeds(seed)

    seed_save_path  = CFG["save_path"].replace(".pth", f"_seed{seed}.pth")
    seed_output_dir = os.path.join(CFG["output_dir"], f"seed_{seed}")
    os.makedirs(os.path.dirname(seed_save_path), exist_ok=True)
    os.makedirs(seed_output_dir, exist_ok=True)

    train_loader = DataLoader(
        FociDataset(CFG["train_csv"], CFG["image_size"], is_train=True),
        batch_size=CFG["batch_size"], shuffle=True,
        collate_fn=safe_collate, num_workers=0
    )
    val_loader = DataLoader(
        FociDataset(CFG["val_csv"], CFG["image_size"], is_train=False),
        batch_size=CFG["batch_size"], shuffle=False,
        collate_fn=safe_collate, num_workers=0
    )

    model = FociViT_Baseline(
        CFG["backbone_name"], CFG["backbone_ckpt"],
        CFG["image_size"], POOLING_MODE
    ).to(DEVICE)

    head_params = list(model.classifier.parameters())
    if POOLING_MODE == "gated_attention":
        head_params += (
            list(model.head_norm.parameters())   +
            list(model.attention_V.parameters()) +
            list(model.attention_U.parameters()) +
            list(model.attention_w.parameters())
        )

    optimizer = torch.optim.AdamW(
        [{"params": head_params, "lr": CFG["lr_head"]}],
        weight_decay=CFG["wd"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG["epochs"], eta_min=1e-6
    )
    criterion = nn.BCEWithLogitsLoss()

    history = {
        "train_loss": [], "val_loss": [],
        "train_acc":  [], "val_acc":  [],
        "val_f1":     [], "lr_head":  []
    }
    best_f1 = 0.0

    for epoch in range(CFG["epochs"]):
        model.train()
        t_loss, t_correct, t_total = 0.0, 0, 0
        for batch in tqdm(train_loader, desc=f"Seed {seed} Epoch {epoch+1} [Train]", leave=False):
            if batch is None: continue
            x, y = [b.to(DEVICE) for b in batch]
            optimizer.zero_grad()
            logits = model(x)
            loss   = criterion(logits, y)
            loss.backward()
            optimizer.step()
            B          = x.size(0)
            t_loss    += loss.item() * B
            t_correct += ((logits >= 0.0).float() == y).sum().item()
            t_total   += B

        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        v_preds_all, v_labels_all  = [], []
        with torch.no_grad():
            for batch in val_loader:
                if batch is None: continue
                x, y   = [b.to(DEVICE) for b in batch]
                logits = model(x)
                loss   = criterion(logits, y)
                B          = x.size(0)
                v_loss    += loss.item() * B
                v_correct += ((logits >= 0.0).float() == y).sum().item()
                v_total   += B
                v_preds_all.extend((logits >= 0.0).float().cpu().numpy())
                v_labels_all.extend(y.cpu().numpy())

        scheduler.step()

        lr_now         = optimizer.param_groups[0]["lr"]
        train_loss_avg = t_loss    / t_total
        val_loss_avg   = v_loss    / max(v_total, 1)
        train_acc_avg  = t_correct / t_total
        val_acc_avg    = v_correct / max(v_total, 1)
        val_f1         = sk_f1(v_labels_all, v_preds_all, zero_division=0)

        history["train_loss"].append(train_loss_avg)
        history["val_loss"].append(val_loss_avg)
        history["train_acc"].append(train_acc_avg)
        history["val_acc"].append(val_acc_avg)
        history["val_f1"].append(val_f1)
        history["lr_head"].append(lr_now)

        print(
            f"Seed {seed} Epoch {epoch+1:02d}: "
            f"TL={train_loss_avg:.4f} TA={train_acc_avg:.4f} | "
            f"VL={val_loss_avg:.4f} VA={val_acc_avg:.4f} VF1={val_f1:.4f}"
        )

        if val_f1 >= best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), seed_save_path)

    # ── Per-seed training plot ───────────────────────────────────────────────
    fig, ax = plt.subplots(1, 3, figsize=(18, 5))
    epochs_range = range(1, CFG["epochs"] + 1)
    ax[0].plot(epochs_range, history["train_loss"], label="Train Loss", marker="o")
    ax[0].plot(epochs_range, history["val_loss"],   label="Val Loss",   marker="o")
    ax[0].set_title(f"BCE Loss — seed {seed}")
    ax[0].legend(); ax[0].grid(True, alpha=0.3)
    ax[1].plot(epochs_range, history["train_acc"], label="Train Acc", marker="o")
    ax[1].plot(epochs_range, history["val_acc"],   label="Val Acc",   marker="o")
    ax[1].plot(epochs_range, history["val_f1"],    label="Val F1",    marker="o", color="green")
    ax[1].set_title(f"Acc & F1 — seed {seed}")
    ax[1].legend(); ax[1].grid(True, alpha=0.3)
    ax[2].plot(epochs_range, history["lr_head"], label="Head LR", marker="o", color="blue")
    ax[2].set_title(f"LR — seed {seed}")
    ax[2].set_yscale("log"); ax[2].legend(); ax[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{seed_output_dir}/training_metrics.png")
    plt.close()

    print(f"Seed {seed} complete. Best Val F1 = {best_f1:.4f}")
    print(f"Checkpoint: {seed_save_path}")
    return {"seed": seed, "best_val_f1": float(best_f1), "ckpt": seed_save_path}


if __name__ == "__main__":
    os.makedirs(os.path.dirname(CFG["save_path"]), exist_ok=True)
    os.makedirs(CFG["output_dir"], exist_ok=True)
    label_counts = pd.read_csv(TRAIN_CSV)["label"].value_counts()
    print(f"Label counts — 0: {label_counts.get(0, 0)}, 1: {label_counts.get(1, 0)}")

    summary = []
    for s in range(N_SEEDS):
        summary.append(train_one_seed(s))

    print("\n" + "="*60)
    print("  Multi-seed training complete")
    print("="*60)
    for r in summary:
        print(f"  seed {r['seed']}: best val F1 = {r['best_val_f1']:.4f}  ->  {r['ckpt']}")

In [ ]:
# =========================================================
# VARIANT TRAINING — CNN baseline + alt-MIL aggregators
# =========================================================
#
# Variants trained here:
#   resnet50_gated   : ResNet-50 (ImageNet, frozen) + gated-attention MIL head
#   vit_dino_mean    : ViT-Tiny + DINO (frozen) + mean pooling head
#   vit_dino_nogate  : ViT-Tiny + DINO (frozen) + single-branch tanh attention
#                      pooling head (ABMIL, no gate)

VARIANTS = [
    {
        "name":           "resnet50_gated",
        "backbone_name":  "resnet50",
        "backbone_ckpt":  None,                  # None ⇒ use ImageNet pretrained via timm
        "use_imagenet":   True,
        "pooling":        "gated_attention",
    },
    {
        "name":           "vit_dino_mean",
        "backbone_name":  CFG["backbone_name"],  # vit_tiny_patch16_384
        "backbone_ckpt":  BACKBONE_CKPT,
        "use_imagenet":   False,
        "pooling":        "mean",
    },
    {
        "name":           "vit_dino_nogate",
        "backbone_name":  CFG["backbone_name"],
        "backbone_ckpt":  BACKBONE_CKPT,
        "use_imagenet":   False,
        "pooling":        "attention_nogate",
    },
]


class FociClassifierVariant(nn.Module):
    """Configurable backbone + pooling for baseline / aggregator ablations."""

    def __init__(self, backbone_name, backbone_ckpt, img_size,
                 pooling_mode, use_imagenet=False):
        super().__init__()
        self.pooling_mode = pooling_mode
        self.backbone_name = backbone_name
        self.is_vit = backbone_name.startswith("vit_")

        if self.is_vit:
            self.backbone = timm.create_model(
                backbone_name, pretrained=False,
                num_classes=0, global_pool="", img_size=img_size,
            )
            if backbone_ckpt is not None and os.path.exists(backbone_ckpt):
                sd = torch.load(backbone_ckpt, map_location="cpu")
                self.backbone.load_state_dict(
                    {k.replace("backbone.", "").replace("vit.", ""): v
                     for k, v in sd.items()},
                    strict=False,
                )
                print(f"  loaded ViT backbone weights from {backbone_ckpt}")
        else:
            self.backbone = timm.create_model(
                backbone_name, pretrained=use_imagenet,
                num_classes=0, global_pool="",
            )
            if use_imagenet:
                print(f"  loaded ImageNet-pretrained CNN: {backbone_name}")

        for p in self.backbone.parameters():
            p.requires_grad = False

        with torch.no_grad():
            tokens = self._extract_tokens(torch.zeros(1, 3, img_size, img_size))
            dim = tokens.shape[-1]

        hidden_dim = 128
        self.head_norm = nn.LayerNorm(dim)

        if pooling_mode == "gated_attention":
            self.attention_V = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Tanh())
            self.attention_U = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Sigmoid())
            self.attention_w = nn.Linear(hidden_dim, 1)
        elif pooling_mode == "attention_nogate":
            self.attention_V = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Tanh())
            self.attention_w = nn.Linear(hidden_dim, 1)
        elif pooling_mode == "mean":
            pass
        else:
            raise ValueError(f"unknown pooling_mode={pooling_mode}")

        self.classifier = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1)
        )

    def _extract_tokens(self, x):
        """Return per-patch tokens (B, N, D) for both ViT and CNN backbones."""
        if self.is_vit:
            feats = self.backbone.forward_features(x)
            if isinstance(feats, dict):
                feats = feats["x"]
            return feats[:, 1:, :]                          # drop CLS, keep patches
        feats = self.backbone.forward_features(x)           # (B, C, H, W)
        B, C, H, W = feats.shape
        return feats.reshape(B, C, H * W).permute(0, 2, 1)  # (B, H*W, C)

    def forward(self, x):
        patches = self._extract_tokens(x)
        normed  = self.head_norm(patches)

        if self.pooling_mode == "gated_attention":
            A = torch.softmax(
                self.attention_w(self.attention_V(normed) * self.attention_U(normed))
                    .squeeze(-1), dim=1)
            pooled = torch.sum(patches * A.unsqueeze(-1), dim=1)
        elif self.pooling_mode == "attention_nogate":
            A = torch.softmax(
                self.attention_w(self.attention_V(normed)).squeeze(-1), dim=1)
            pooled = torch.sum(patches * A.unsqueeze(-1), dim=1)
        else:                                               # mean
            pooled = normed.mean(dim=1)

        return self.classifier(pooled).squeeze(-1)

    def trainable_head_params(self):
        params = list(self.classifier.parameters()) + list(self.head_norm.parameters())
        for attr in ("attention_V", "attention_U", "attention_w"):
            if hasattr(self, attr):
                params += list(getattr(self, attr).parameters())
        return params


def variant_save_path(variant_name, seed):
    return os.path.join(MODELS_DIR, variant_name, f"baseline_model_seed{seed}.pth")


def train_variant_seed(variant, seed):
    name = variant["name"]
    print("\n" + "#"*60)
    print(f"  Variant {name} | seed {seed}")
    print("#"*60)

    set_all_seeds(seed)
    save_path = variant_save_path(name, seed)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    train_loader = DataLoader(
        FociDataset(CFG["train_csv"], CFG["image_size"], is_train=True),
        batch_size=CFG["batch_size"], shuffle=True,
        collate_fn=safe_collate, num_workers=0,
    )
    val_loader = DataLoader(
        FociDataset(CFG["val_csv"], CFG["image_size"], is_train=False),
        batch_size=CFG["batch_size"], shuffle=False,
        collate_fn=safe_collate, num_workers=0,
    )

    model = FociClassifierVariant(
        backbone_name=variant["backbone_name"],
        backbone_ckpt=variant["backbone_ckpt"],
        img_size=CFG["image_size"],
        pooling_mode=variant["pooling"],
        use_imagenet=variant["use_imagenet"],
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        [{"params": model.trainable_head_params(), "lr": CFG["lr_head"]}],
        weight_decay=CFG["wd"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG["epochs"], eta_min=1e-6,
    )
    criterion = nn.BCEWithLogitsLoss()
    best_f1 = 0.0

    for epoch in range(CFG["epochs"]):
        model.train()
        for batch in tqdm(train_loader, desc=f"{name} s{seed} ep{epoch+1}", leave=False):
            if batch is None: continue
            x, y = [b.to(DEVICE) for b in batch]
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                if batch is None: continue
                x, y = [b.to(DEVICE) for b in batch]
                logits = model(x)
                preds.extend((logits >= 0.0).float().cpu().numpy())
                labels.extend(y.cpu().numpy())
        scheduler.step()

        val_f1 = sk_f1(labels, preds, zero_division=0)
        print(f"  {name} s{seed} ep{epoch+1:02d}: val F1 = {val_f1:.4f}")
        if val_f1 >= best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), save_path)

    print(f"  done {name} s{seed}: best val F1 = {best_f1:.4f}  ->  {save_path}")
    return {"variant": name, "seed": seed, "best_val_f1": float(best_f1),
            "ckpt": save_path}


if __name__ == "__main__":
    print("\nVariants to train:")
    for v in VARIANTS:
        print(f"  - {v['name']}  ({v['backbone_name']}, pooling={v['pooling']})")

    summary = []
    for v in VARIANTS:
        for s in range(N_SEEDS):
            sp = variant_save_path(v["name"], s)
            if os.path.exists(sp):
                print(f"\n[skip] {v['name']} seed {s} — already trained at {sp}")
                continue
            summary.append(train_variant_seed(v, s))

    print("\n" + "="*60)
    print("  Variant training complete")
    print("="*60)
    for r in summary:
        print(f"  {r['variant']} seed {r['seed']}: F1={r['best_val_f1']:.4f}")

# **Evaluation**

In [ ]:
import os
import json
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import timm
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc,
    average_precision_score, precision_recall_curve,
)

# =========================
# CONFIGURATION
# =========================
SIZE_MODE    = "384"
POOLING_MODE = "gated_attention"

# Recall floor for val-tuned operating point (drives FN minimisation).
TARGET_RECALL = 0.95

# Reproducibility for bootstrap.
BOOTSTRAP_SEED = 12345
N_BOOTSTRAP    = 1000

# Number of seeds to pull from the multi-seed training run.
N_SEEDS = 5

SAVE_VISUALIZATIONS = True
NUM_VIS_TO_SAVE     = 30   # rendered for seed 0 only, just as a sanity check

# ── Writable root for this revision (must match cell-1's PAPER_REV_ROOT) ─────
PAPER_REV_ROOT = "/content/drive/MyDrive/FYP/paper_revision_v01"
CKPT_DIR       = os.path.join(PAPER_REV_ROOT, "models")
CKPT_TEMPLATE  = os.path.join(CKPT_DIR, "baseline_model_seed{seed}.pth")
OUTPUT_DIR     = os.path.join(PAPER_REV_ROOT, "eval_results")

# ── Read-only data CSVs ──────────────────────────────────────────────────────
VAL_CSV  = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/val_dataset.csv"
TEST_CSV = "/content/drive/MyDrive/FYP/natural foci ground truths/data/culture_only_v02/test_dataset.csv"

CFG = {
    "batch_size":    32,
    "val_csv":       VAL_CSV,
    "test_csv":      TEST_CSV,
    "image_size":    int(SIZE_MODE),
    "patch_size":    16,
    "backbone_name": f"vit_tiny_patch16_{SIZE_MODE}",
    "device":        torch.device("cuda" if torch.cuda.is_available() else "cpu"),
}

GRID_SIZE = CFG["image_size"] // CFG["patch_size"]   # 24
DEVICE    = CFG["device"]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"--- PAPER-NUMBERS EVAL: {POOLING_MODE.upper()} ---")
print(f"Writable root: {PAPER_REV_ROOT}")
print(f"Ckpt template: {CKPT_TEMPLATE}")
print(f"N seeds      : {N_SEEDS}")
print(f"Val CSV      : {CFG['val_csv']}")
print(f"Test CSV     : {CFG['test_csv']}")
print(f"Output dir   : {OUTPUT_DIR}")

# =========================
# DATASETS
# =========================
NORM_MEAN = [0.2251, 0.2251, 0.2251]
NORM_STD  = [0.2375, 0.2375, 0.2375]

eval_transform = transforms.Compose([
    transforms.Resize((CFG["image_size"], CFG["image_size"]),
                      interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

class FociEvalDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img_p = str(row["image_path"])
            img   = Image.open(img_p).convert("RGB")
            x     = eval_transform(img)
            y     = float(row["label"])
            return x, torch.tensor(y).float(), img_p
        except Exception: return None

def safe_collate_eval(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return None
    xs, ys, paths = zip(*batch)
    return torch.stack(xs), torch.stack(ys), list(paths)

# =========================
# EVAL MODEL
# =========================
class FociViT_Eval(nn.Module):
    def __init__(self, backbone_name, img_size=384, pooling_mode="gated_attention"):
        super().__init__()
        self.pooling_mode = pooling_mode
        self.backbone = timm.create_model(
            backbone_name, pretrained=False,
            num_classes=0, global_pool="", img_size=img_size
        )
        with torch.no_grad():
            dim = self.backbone.forward_features(
                torch.zeros(1, 3, img_size, img_size)
            ).shape[-1]
        if self.pooling_mode == "gated_attention":
            hidden_dim = 128
            self.head_norm   = nn.LayerNorm(dim)
            self.attention_V = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Tanh())
            self.attention_U = nn.Sequential(nn.Linear(dim, hidden_dim), nn.Sigmoid())
            self.attention_w = nn.Linear(hidden_dim, 1)
        self.classifier = nn.Sequential(
            nn.Linear(dim, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 1)
        )

    def forward(self, x):
        feats = self.backbone.forward_features(x)
        if isinstance(feats, dict): feats = feats["x"]
        patch_tokens = feats[:, 1:, :]
        spatial_map  = None
        if self.pooling_mode == "gated_attention":
            normed_patches = self.head_norm(patch_tokens)
            A_V   = self.attention_V(normed_patches)
            A_U   = self.attention_U(normed_patches)
            A_raw = self.attention_w(A_V * A_U).squeeze(-1)
            A     = torch.softmax(A_raw, dim=1)
            pooled_features = torch.sum(patch_tokens * A.unsqueeze(-1), dim=1)
            spatial_map     = A
        else:
            pooled_features = feats[:, 0, :]
        logits = self.classifier(pooled_features).squeeze(-1)
        probs  = torch.sigmoid(logits)
        return probs, spatial_map, patch_tokens

# =========================
# DASHBOARD (seed 0 only)
# =========================
def norm_01(arr):
    return (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

def save_dashboard(img_path, patch_tokens, attn_map, prob, label, idx, out_dir, threshold=0.5):
    pil_img     = Image.open(img_path).convert("RGB")
    img_resized = pil_img.resize((CFG["image_size"], CFG["image_size"]))
    tokens_np   = patch_tokens.cpu().numpy()
    gt_label    = "Damaged"  if int(label) == 1 else "Healthy"
    pred_label  = "Damaged"  if prob >= threshold else "Healthy"
    correct     = "✓" if pred_label == gt_label else "✗"
    norms    = np.linalg.norm(tokens_np, axis=-1)
    norm_map = norm_01(norms.reshape(GRID_SIZE, GRID_SIZE))
    pca     = PCA(n_components=3)
    pca_out = pca.fit_transform(tokens_np)
    for c in range(3): pca_out[:, c] = norm_01(pca_out[:, c])
    pca_rgb = pca_out.reshape(GRID_SIZE, GRID_SIZE, 3)
    attn_2d  = norm_01(attn_map.cpu().numpy().reshape(GRID_SIZE, GRID_SIZE))
    attn_img = Image.fromarray((attn_2d * 255).astype(np.uint8)).resize(
        (CFG["image_size"], CFG["image_size"]), Image.BILINEAR
    )
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes[0, 0].imshow(img_resized); axes[0, 0].set_title(f"Original  |  GT: {gt_label}", fontsize=10)
    im1 = axes[0, 1].imshow(norm_map, cmap="magma"); axes[0, 1].set_title("Backbone: Patch Token L2 Norm", fontsize=10)
    fig.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)
    axes[0, 2].imshow(pca_rgb); axes[0, 2].set_title("Backbone: PCA RGB Map", fontsize=10)
    axes[1, 0].imshow(img_resized); axes[1, 0].set_title(f"Pred: {pred_label} ({prob:.3f}) {correct}", fontsize=10)
    im2 = axes[1, 1].imshow(attn_2d, cmap="jet", vmin=0, vmax=1); axes[1, 1].set_title("Head: Gated Attention Map", fontsize=10)
    fig.colorbar(im2, ax=axes[1, 1], fraction=0.046, pad=0.04)
    axes[1, 2].imshow(img_resized); axes[1, 2].imshow(attn_img, cmap="jet", alpha=0.5)
    axes[1, 2].set_title("Head: Attention Overlay", fontsize=10)
    for ax in axes.flatten(): ax.axis("off")
    plt.suptitle(f"{POOLING_MODE.upper()} | Sample {idx}", fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{out_dir}/dashboard_{idx}_gt{int(label)}.png", bbox_inches="tight")
    plt.close(fig)

# =========================
# INFERENCE HELPERS
# =========================
def infer_split(model, csv_path, collect_visuals=False):
    """Run inference on a CSV split. Returns (y_true, y_probs, paths, visuals_or_None)."""
    loader = DataLoader(
        FociEvalDataset(csv_path),
        batch_size=CFG["batch_size"], shuffle=False,
        collate_fn=safe_collate_eval, num_workers=0
    )
    all_probs, all_labels, all_paths = [], [], []
    visuals = [] if collect_visuals else None
    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            if batch is None: continue
            x, y, paths = batch
            x, y = x.to(DEVICE), y.to(DEVICE)
            probs, spatial_maps, patch_tokens_batch = model(x)
            all_probs.extend(probs.cpu().numpy().tolist())
            all_labels.extend(y.cpu().numpy().tolist())
            all_paths.extend(paths)
            if collect_visuals and spatial_maps is not None and len(visuals) < NUM_VIS_TO_SAVE:
                for i in range(x.size(0)):
                    if len(visuals) >= NUM_VIS_TO_SAVE: break
                    visuals.append((
                        paths[i], patch_tokens_batch[i].cpu(),
                        spatial_maps[i].cpu(), float(probs[i].item()), float(y[i].item())
                    ))
    return (np.array(all_labels).astype(int),
            np.array(all_probs).astype(float),
            all_paths, visuals)


def pick_target_recall_threshold(y_true_val, y_probs_val, target_recall):
    """Highest threshold on val with TPR >= target_recall. Bounds FN rate."""
    fpr, tpr, thr = roc_curve(y_true_val, y_probs_val)
    valid = np.where(tpr >= target_recall)[0]
    if len(valid):
        idx = int(valid[0])    # roc_curve returns thr descending → first hit is highest thr
    else:
        idx = int(np.argmax(tpr))
    return float(thr[idx]), float(tpr[idx]), float(fpr[idx])


def metrics_at_threshold(y_true, y_probs, tau):
    yh = (y_probs >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yh, labels=[0, 1]).ravel()
    acc = accuracy_score(y_true, yh)
    prec = precision_score(y_true, yh, zero_division=0)
    rec  = recall_score(y_true, yh, zero_division=0)
    f1   = f1_score(y_true, yh, zero_division=0)
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn),
        "accuracy": float(acc), "precision": float(prec),
        "recall": float(rec), "f1": float(f1), "specificity": float(spec),
    }


def roc_auc_safe(y_true, y_probs):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    return float(auc(fpr, tpr))


def bootstrap_test_metrics(y_true, y_probs, tau, n_boot, rng):
    """Test-set bootstrap CIs at fixed τ. Returns dict of per-metric value arrays."""
    n = len(y_true)
    keys = ["accuracy", "precision", "recall", "f1", "specificity",
            "roc_auc", "pr_auc"]
    out = {k: [] for k in keys}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_probs[idx]
        if len(np.unique(yt)) < 2:    # degenerate resample, skip
            continue
        m = metrics_at_threshold(yt, yp, tau)
        try:
            roc_a = roc_auc_safe(yt, yp)
            pr_a  = average_precision_score(yt, yp)
        except Exception:
            continue
        out["accuracy"].append(m["accuracy"])
        out["precision"].append(m["precision"])
        out["recall"].append(m["recall"])
        out["f1"].append(m["f1"])
        out["specificity"].append(m["specificity"])
        out["roc_auc"].append(roc_a)
        out["pr_auc"].append(pr_a)
    return {k: np.array(v) for k, v in out.items()}


def agg_stats(values):
    """mean, SD, and seed-based 95% normal CI."""
    v = np.array(values, dtype=float)
    mean = float(np.mean(v))
    sd   = float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
    se   = sd / np.sqrt(max(len(v), 1))
    return {
        "mean": mean, "sd": sd,
        "ci_seeds_low":  mean - 1.96 * se,
        "ci_seeds_high": mean + 1.96 * se,
    }


def boot_ci(pooled_values):
    if len(pooled_values) == 0:
        return [float("nan"), float("nan")]
    lo, hi = np.percentile(pooled_values, [2.5, 97.5])
    return [float(lo), float(hi)]


# =========================
# STAGE A — per-seed inference on val + test
# =========================
print("\n>>> STAGE A — per-seed inference\n")
seed_data = {}   # seed -> dict of arrays + paths

for seed in range(N_SEEDS):
    ckpt = CKPT_TEMPLATE.format(seed=seed)
    if not os.path.exists(ckpt):
        raise FileNotFoundError(
            f"Missing checkpoint for seed {seed}: {ckpt}\n"
            f"Run the training cell first."
        )
    print(f"Seed {seed}: loading {ckpt}")
    model = FociViT_Eval(CFG["backbone_name"], CFG["image_size"], POOLING_MODE).to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE), strict=False)
    model.eval()

    y_val,  p_val,  paths_val,  _        = infer_split(model, CFG["val_csv"],  collect_visuals=False)
    y_test, p_test, paths_test, visuals  = infer_split(
        model, CFG["test_csv"], collect_visuals=(seed == 0 and SAVE_VISUALIZATIONS)
    )

    seed_dir = os.path.join(OUTPUT_DIR, f"seed_{seed}")
    os.makedirs(seed_dir, exist_ok=True)

    pd.DataFrame({"image_path": paths_val,  "label": y_val,  "prob": p_val}) \
        .to_csv(os.path.join(seed_dir, "predictions_val.csv"), index=False)
    pd.DataFrame({"image_path": paths_test, "label": y_test, "prob": p_test}) \
        .to_csv(os.path.join(seed_dir, "predictions_test.csv"), index=False)

    seed_data[seed] = {
        "y_val": y_val,  "p_val": p_val,
        "y_test": y_test, "p_test": p_test,
        "visuals": visuals,
    }
    print(f"  val: {len(y_val)} samples | test: {len(y_test)} samples")

# =========================
# STAGE B — pick τ on val per seed (target recall)
# =========================
print("\n>>> STAGE B — threshold selection on validation\n")
thresholds = {}
for seed in range(N_SEEDS):
    d = seed_data[seed]
    tau_v, tpr_v, fpr_v = pick_target_recall_threshold(d["y_val"], d["p_val"], TARGET_RECALL)
    thresholds[seed] = {
        "tau_targetrecall_val": tau_v,
        "tau_fixed":            0.5,
        "val_tpr_at_tau":       tpr_v,
        "val_fpr_at_tau":       fpr_v,
        "target_recall":        TARGET_RECALL,
    }
    print(f"  seed {seed}: τ_val = {tau_v:.4f}  (val TPR={tpr_v:.3f}, val FPR={fpr_v:.3f})")

with open(os.path.join(OUTPUT_DIR, "thresholds.json"), "w") as f:
    json.dump(thresholds, f, indent=2)

# =========================
# STAGE C — per-seed test metrics + bootstrap CIs
# =========================
print("\n>>> STAGE C — test metrics + bootstrap per seed\n")
rng = np.random.default_rng(BOOTSTRAP_SEED)
per_seed = []

for seed in range(N_SEEDS):
    d = seed_data[seed]
    y, p = d["y_test"], d["p_test"]
    tau_v = thresholds[seed]["tau_targetrecall_val"]

    roc_a = roc_auc_safe(y, p)
    pr_a  = float(average_precision_score(y, p))
    m_05  = metrics_at_threshold(y, p, 0.5)
    m_val = metrics_at_threshold(y, p, tau_v)

    boot_05  = bootstrap_test_metrics(y, p, 0.5,    N_BOOTSTRAP, rng)
    boot_val = bootstrap_test_metrics(y, p, tau_v,  N_BOOTSTRAP, rng)

    per_seed.append({
        "seed":          seed,
        "tau_val":       tau_v,
        "roc_auc":       roc_a,
        "pr_auc":        pr_a,
        "at_tau_fixed":  m_05,
        "at_tau_val":    m_val,
        "boot_at_tau_fixed": {k: v.tolist() for k, v in boot_05.items()},
        "boot_at_tau_val":   {k: v.tolist() for k, v in boot_val.items()},
    })
    print(f"  seed {seed}: ROC-AUC={roc_a:.4f}  PR-AUC={pr_a:.4f}  "
          f"F1@0.5={m_05['f1']:.3f}  F1@τ_val={m_val['f1']:.3f}  "
          f"FN@τ_val={m_val['fn']}")

# =========================
# STAGE D — aggregation across seeds
# =========================
print("\n>>> STAGE D — aggregation across seeds\n")

def collect(field):
    return [s[field] for s in per_seed]

def collect_op(op_key, metric):
    return [s[op_key][metric] for s in per_seed]

def pool_boot(op_key, metric):
    pooled = []
    for s in per_seed:
        pooled.extend(s[op_key][metric])
    return pooled

headline = {
    "roc_auc": {
        **agg_stats(collect("roc_auc")),
        "ci_bootstrap": boot_ci(pool_boot("boot_at_tau_val", "roc_auc")),
        "per_seed": collect("roc_auc"),
    },
    "pr_auc": {
        **agg_stats(collect("pr_auc")),
        "ci_bootstrap": boot_ci(pool_boot("boot_at_tau_val", "pr_auc")),
        "per_seed": collect("pr_auc"),
    },
}

def aggregate_op(op_key, label_for_tau):
    block = {}
    if op_key == "at_tau_val":
        block["tau"] = agg_stats(collect("tau_val"))
    else:
        block["tau"] = {"mean": 0.5, "sd": 0.0,
                         "ci_seeds_low": 0.5, "ci_seeds_high": 0.5}
    for metric in ["accuracy", "precision", "recall", "f1", "specificity"]:
        block[metric] = {
            **agg_stats(collect_op(op_key, metric)),
            "ci_bootstrap": boot_ci(pool_boot(f"boot_{op_key}", metric)),
            "per_seed": collect_op(op_key, metric),
        }
    for c in ["tp", "fp", "fn", "tn"]:
        vals = collect_op(op_key, c)
        block[c] = {
            "mean": float(np.mean(vals)),
            "sd":   float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
            "per_seed": vals,
        }
    block["label"] = label_for_tau
    return block

operating_points = {
    "tau_fixed_0.5":         aggregate_op("at_tau_fixed", "τ = 0.5 (reference)"),
    "tau_targetrecall_val":  aggregate_op("at_tau_val",
        f"τ tuned on val s.t. recall ≥ {TARGET_RECALL:.2f}"),
}

# Strip bulky per-seed bootstrap arrays from the JSON output (kept in memory only).
per_seed_lean = []
for s in per_seed:
    s2 = {k: v for k, v in s.items() if not k.startswith("boot_")}
    per_seed_lean.append(s2)

paper_numbers = {
    "config": {
        "n_seeds":        N_SEEDS,
        "target_recall":  TARGET_RECALL,
        "n_bootstrap":    N_BOOTSTRAP,
        "bootstrap_seed": BOOTSTRAP_SEED,
        "ckpt_template":  CKPT_TEMPLATE,
        "val_csv":        VAL_CSV,
        "test_csv":       TEST_CSV,
    },
    "headline_metrics": headline,
    "operating_points": operating_points,
    "per_seed":         per_seed_lean,
    "thresholds":       thresholds,
}

with open(os.path.join(OUTPUT_DIR, "paper_numbers.json"), "w") as f:
    json.dump(paper_numbers, f, indent=2)
print(f"  wrote {os.path.join(OUTPUT_DIR, 'paper_numbers.json')}")

# =========================
# STAGE E — figures
# =========================
print("\n>>> STAGE E — figures\n")

plt.figure(figsize=(6, 6))
for s in per_seed:
    seed = s["seed"]
    y, p = seed_data[seed]["y_test"], seed_data[seed]["p_test"]
    prec_c, rec_c, _ = precision_recall_curve(y, p)
    plt.plot(rec_c, prec_c, alpha=0.6, lw=1.5,
             label=f"seed {seed}: AP = {s['pr_auc']:.3f}")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title(f"Precision–Recall — {N_SEEDS} seeds  "
          f"(AP = {headline['pr_auc']['mean']:.3f} ± {headline['pr_auc']['sd']:.3f})")
plt.legend(loc="lower left", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pr_curve_seeds.png"))
plt.close()

plt.figure(figsize=(6, 6))
for s in per_seed:
    seed = s["seed"]
    y, p = seed_data[seed]["y_test"], seed_data[seed]["p_test"]
    fpr_c, tpr_c, _ = roc_curve(y, p)
    plt.plot(fpr_c, tpr_c, alpha=0.6, lw=1.5,
             label=f"seed {seed}: AUC = {s['roc_auc']:.3f}")
plt.plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--", label="chance")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title(f"ROC — {N_SEEDS} seeds  "
          f"(AUC = {headline['roc_auc']['mean']:.3f} ± {headline['roc_auc']['sd']:.3f})")
plt.legend(loc="lower right", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "roc_curve_seeds.png"))
plt.close()

def plot_cm(op_block, fname, title_suffix):
    mean = np.array([
        [op_block["tn"]["mean"], op_block["fp"]["mean"]],
        [op_block["fn"]["mean"], op_block["tp"]["mean"]],
    ])
    sd = np.array([
        [op_block["tn"]["sd"], op_block["fp"]["sd"]],
        [op_block["fn"]["sd"], op_block["tp"]["sd"]],
    ])
    annot = np.empty_like(mean, dtype=object)
    for i in range(2):
        for j in range(2):
            annot[i, j] = f"{mean[i,j]:.1f}\n±{sd[i,j]:.1f}"
    plt.figure(figsize=(5, 4))
    sns.heatmap(mean, annot=annot, fmt="", cmap="Blues",
                xticklabels=["Healthy", "Damaged"],
                yticklabels=["Healthy", "Damaged"])
    plt.xlabel("Predicted"); plt.ylabel("Ground Truth")
    plt.title(f"Confusion Matrix — {title_suffix}\n(mean ± SD across {N_SEEDS} seeds)")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname))
    plt.close()

plot_cm(operating_points["tau_fixed_0.5"],
        "confusion_matrix_tau050.png", "τ = 0.5")
plot_cm(operating_points["tau_targetrecall_val"],
        "confusion_matrix_tau_val.png",
        f"τ on val s.t. recall ≥ {TARGET_RECALL:.2f}")

if SAVE_VISUALIZATIONS and seed_data[0]["visuals"]:
    seed0_dir = os.path.join(OUTPUT_DIR, "seed_0_dashboards")
    os.makedirs(seed0_dir, exist_ok=True)
    tau0 = thresholds[0]["tau_targetrecall_val"]
    for idx, (path, pt, am, prob, label) in enumerate(seed_data[0]["visuals"]):
        save_dashboard(path, pt, am, prob, label, idx, seed0_dir, threshold=tau0)



In [ ]:
# =========================================================
# COMBINED VARIANT EVALUATION — proposed + new baselines + alt-MIL
# =========================================================


VARIANTS_EVAL = [
    {
        "name":           "proposed",
        "display":        "Proposed (ViT-Tiny + DINO + gated)",
        "backbone_name":  CFG["backbone_name"],
        "backbone_ckpt":  None,                        # head only; backbone init irrelevant for state_dict load
        "use_imagenet":   False,
        "pooling":        "gated_attention",
        "ckpt_template":  os.path.join(CKPT_DIR, "baseline_model_seed{seed}.pth"),
    },
    {
        "name":           "resnet50_gated",
        "display":        "ResNet-50 + gated (ImageNet)",
        "backbone_name":  "resnet50",
        "backbone_ckpt":  None,
        "use_imagenet":   False,                       # state_dict load supplies backbone weights
        "pooling":        "gated_attention",
        "ckpt_template":  os.path.join(CKPT_DIR, "resnet50_gated", "baseline_model_seed{seed}.pth"),
    },
    {
        "name":           "vit_dino_mean",
        "display":        "ViT-Tiny + DINO + mean pooling",
        "backbone_name":  CFG["backbone_name"],
        "backbone_ckpt":  None,
        "use_imagenet":   False,
        "pooling":        "mean",
        "ckpt_template":  os.path.join(CKPT_DIR, "vit_dino_mean", "baseline_model_seed{seed}.pth"),
    },
    {
        "name":           "vit_dino_nogate",
        "display":        "ViT-Tiny + DINO + ABMIL (no gate)",
        "backbone_name":  CFG["backbone_name"],
        "backbone_ckpt":  None,
        "use_imagenet":   False,
        "pooling":        "attention_nogate",
        "ckpt_template":  os.path.join(CKPT_DIR, "vit_dino_nogate", "baseline_model_seed{seed}.pth"),
    },
]

VARIANTS_OUTPUT_DIR = os.path.join(PAPER_REV_ROOT, "eval_results_variants")
os.makedirs(VARIANTS_OUTPUT_DIR, exist_ok=True)
print(f"--- COMBINED VARIANT EVAL ---")
print(f"Output dir : {VARIANTS_OUTPUT_DIR}")
print(f"N seeds    : {N_SEEDS}")
for v in VARIANTS_EVAL:
    print(f"  - {v['name']}: {v['ckpt_template']}")


def infer_variant_seed(model, csv_path):
    """Variant inference returning (y_true, y_probs, paths)."""
    loader = DataLoader(
        FociEvalDataset(csv_path),
        batch_size=CFG["batch_size"], shuffle=False,
        collate_fn=safe_collate_eval, num_workers=0,
    )
    all_probs, all_labels, all_paths = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            if batch is None: continue
            x, y, paths = batch
            x = x.to(DEVICE)
            logits = model(x)
            probs = torch.sigmoid(logits)
            all_probs.extend(probs.cpu().numpy().tolist())
            all_labels.extend(y.cpu().numpy().tolist())
            all_paths.extend(paths)
    return (np.array(all_labels).astype(int),
            np.array(all_probs).astype(float),
            all_paths)


def evaluate_variant(variant):
    """Full 5-seed eval for a variant. Returns aggregated metrics dict."""
    name = variant["name"]
    print(f"\n>>> Variant: {name}  ({variant['display']})")
    rng = np.random.default_rng(BOOTSTRAP_SEED)

    per_seed_arrs = []     # for downstream curves
    per_seed_metr = []
    thresholds_v = {}

    for seed in range(N_SEEDS):
        ckpt = variant["ckpt_template"].format(seed=seed)
        if not os.path.exists(ckpt):
            raise FileNotFoundError(
                f"Missing checkpoint for variant '{name}' seed {seed}: {ckpt}\n"
                "Run the variant training cell first."
            )

        model = FociClassifierVariant(
            backbone_name=variant["backbone_name"],
            backbone_ckpt=variant["backbone_ckpt"],
            img_size=CFG["image_size"],
            pooling_mode=variant["pooling"],
            use_imagenet=variant["use_imagenet"],
        ).to(DEVICE)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE), strict=False)
        model.eval()

        y_v, p_v, _ = infer_variant_seed(model, CFG["val_csv"])
        y_t, p_t, _ = infer_variant_seed(model, CFG["test_csv"])

        tau_v, tpr_v, fpr_v = pick_target_recall_threshold(y_v, p_v, TARGET_RECALL)
        thresholds_v[seed] = {
            "tau_targetrecall_val": tau_v,
            "val_tpr_at_tau":       tpr_v,
            "val_fpr_at_tau":       fpr_v,
        }

        roc_a = roc_auc_safe(y_t, p_t)
        pr_a  = float(average_precision_score(y_t, p_t))
        m_05  = metrics_at_threshold(y_t, p_t, 0.5)
        m_val = metrics_at_threshold(y_t, p_t, tau_v)
        boot_05  = bootstrap_test_metrics(y_t, p_t, 0.5,   N_BOOTSTRAP, rng)
        boot_val = bootstrap_test_metrics(y_t, p_t, tau_v, N_BOOTSTRAP, rng)

        per_seed_metr.append({
            "seed":          seed,
            "tau_val":       tau_v,
            "roc_auc":       roc_a,
            "pr_auc":        pr_a,
            "at_tau_fixed":  m_05,
            "at_tau_val":    m_val,
            "boot_at_tau_fixed": {k: v.tolist() for k, v in boot_05.items()},
            "boot_at_tau_val":   {k: v.tolist() for k, v in boot_val.items()},
        })
        per_seed_arrs.append({"seed": seed, "y_test": y_t, "p_test": p_t})

        print(f"  s{seed}: ROC-AUC={roc_a:.4f}  PR-AUC={pr_a:.4f}  "
              f"F1@0.5={m_05['f1']:.3f}  F1@τ_val={m_val['f1']:.3f}  "
              f"FN@τ_val={m_val['fn']}")

    # ---- aggregate over seeds ----
    def collect(field):
        return [s[field] for s in per_seed_metr]

    def collect_op(op_key, metric):
        return [s[op_key][metric] for s in per_seed_metr]

    def pool_boot(op_key, metric):
        pooled = []
        for s in per_seed_metr:
            pooled.extend(s[op_key][metric])
        return pooled

    headline = {
        "roc_auc": {**agg_stats(collect("roc_auc")),
                    "ci_bootstrap": boot_ci(pool_boot("boot_at_tau_val", "roc_auc"))},
        "pr_auc":  {**agg_stats(collect("pr_auc")),
                    "ci_bootstrap": boot_ci(pool_boot("boot_at_tau_val", "pr_auc"))},
    }

    def aggregate_op(op_key):
        block = {}
        if op_key == "at_tau_val":
            block["tau"] = agg_stats(collect("tau_val"))
        else:
            block["tau"] = {"mean": 0.5, "sd": 0.0,
                             "ci_seeds_low": 0.5, "ci_seeds_high": 0.5}
        for metric in ["accuracy", "precision", "recall", "f1", "specificity"]:
            block[metric] = {
                **agg_stats(collect_op(op_key, metric)),
                "ci_bootstrap": boot_ci(pool_boot(f"boot_{op_key}", metric)),
            }
        for c in ["tp", "fp", "fn", "tn"]:
            vals = collect_op(op_key, c)
            block[c] = {
                "mean": float(np.mean(vals)),
                "sd":   float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
            }
        return block

    return {
        "variant":     name,
        "display":     variant["display"],
        "headline":    headline,
        "at_tau_05":   aggregate_op("at_tau_fixed"),
        "at_tau_val":  aggregate_op("at_tau_val"),
        "thresholds":  thresholds_v,
        "per_seed":    per_seed_arrs,        # arrays for curves
    }


# =========================
# Run eval for all variants
# =========================
results = []
for variant in VARIANTS_EVAL:
    results.append(evaluate_variant(variant))

# =========================
# Combined JSON (without raw arrays)
# =========================
results_lean = []
for r in results:
    results_lean.append({k: v for k, v in r.items() if k != "per_seed"})

with open(os.path.join(VARIANTS_OUTPUT_DIR, "paper_numbers_variants.json"), "w") as f:
    json.dump({
        "config": {
            "n_seeds":       N_SEEDS,
            "target_recall": TARGET_RECALL,
            "n_bootstrap":   N_BOOTSTRAP,
            "val_csv":       VAL_CSV,
            "test_csv":      TEST_CSV,
        },
        "variants": results_lean,
    }, f, indent=2)
print(f"\nwrote {os.path.join(VARIANTS_OUTPUT_DIR, 'paper_numbers_variants.json')}")

# =========================
# Overlay PR and ROC plots (one mean-precision-recall curve per variant)
# =========================
def interp_pr_curve(y_true, y_probs, recall_grid):
    """Interpolate precision onto a fixed recall grid (decreasing in thr)."""
    prec, rec, _ = precision_recall_curve(y_true, y_probs)
    order = np.argsort(rec)
    return np.interp(recall_grid, rec[order], prec[order])

def interp_roc_curve(y_true, y_probs, fpr_grid):
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    order = np.argsort(fpr)
    return np.interp(fpr_grid, fpr[order], tpr[order])

recall_grid = np.linspace(0, 1, 200)
fpr_grid    = np.linspace(0, 1, 200)
colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

plt.figure(figsize=(6.5, 6))
for r, c in zip(results, colors):
    precs = np.stack([interp_pr_curve(s["y_test"], s["p_test"], recall_grid)
                      for s in r["per_seed"]])
    mean_p = precs.mean(axis=0)
    auc_str = f"{r['headline']['pr_auc']['mean']:.3f}"
    sd_str  = f"{r['headline']['pr_auc']['sd']:.3f}"
    plt.plot(recall_grid, mean_p, lw=2, color=c,
             label=f"{r['display']}: AP = {auc_str} ± {sd_str}")
plt.xlabel("Recall"); plt.ylabel("Precision")
plt.title(f"PR curve comparison across variants ({N_SEEDS} seeds, mean curves)")
plt.legend(loc="lower left", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(VARIANTS_OUTPUT_DIR, "variants_pr_curve.png"))
plt.close()

plt.figure(figsize=(6.5, 6))
for r, c in zip(results, colors):
    tprs = np.stack([interp_roc_curve(s["y_test"], s["p_test"], fpr_grid)
                     for s in r["per_seed"]])
    mean_t = tprs.mean(axis=0)
    auc_str = f"{r['headline']['roc_auc']['mean']:.3f}"
    sd_str  = f"{r['headline']['roc_auc']['sd']:.3f}"
    plt.plot(fpr_grid, mean_t, lw=2, color=c,
             label=f"{r['display']}: AUC = {auc_str} ± {sd_str}")
plt.plot([0, 1], [0, 1], color="black", lw=1, linestyle="--", label="chance")
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title(f"ROC curve comparison across variants ({N_SEEDS} seeds, mean curves)")
plt.legend(loc="lower right", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(VARIANTS_OUTPUT_DIR, "variants_roc_curve.png"))
plt.close()
